# 02 - Bronze ingestion

Automatiza la carga desde Parquet canónico guardado en el volumen hacia la tabla Bronze Delta.

Este notebook calcula `source_record_hash` dentro de Bronze para controlar idempotencia. Así no depende de que las fuentes Hugging Face, Kaggle o archivos manuales traigan esa columna.

La entrada principal ya no es una carga manual de CSV. Primero se ejecuta `01_download_source_parquets`, que deja Parquet en el volumen; luego este notebook carga Bronze desde esos Parquet.


In [0]:
from datetime import datetime
from pyspark.sql import functions as F

In [0]:
try:
    dbutils.widgets.text("catalog", "workspace")
    dbutils.widgets.text("schema", "financial_sentiment")
    dbutils.widgets.text("volume", "raw_landing")
    dbutils.widgets.text("create_demo_if_empty", "false")
    dbutils.widgets.text("source_mode", "canonical_parquet")  # canonical_parquet | csv_landing
except Exception:
    pass

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
create_demo_if_empty = dbutils.widgets.get("create_demo_if_empty").lower() == "true"
source_mode = dbutils.widgets.get("source_mode")

full_schema = f"{catalog}.{schema}"
volume_root = f"/Volumes/{catalog}/{schema}/{volume}"
landing_path = f"{volume_root}/incoming"
canonical_root = f"{volume_root}/sources/processed/canonical"
bronze_table = f"{full_schema}.bronze_financial_sentiment_raw"
audit_table = f"{full_schema}.ingestion_file_audit"
run_log_table = f"{full_schema}.pipeline_run_log"

batch_id = f"bronze_{datetime.utcnow().strftime('%Y%m%d%H%M%S')}"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {full_schema}")
try:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {full_schema}.{volume}")
except Exception as e:
    print("El volumen no pudo crearse desde el notebook. Verifica permisos o créalo desde Catalog Explorer.")
    print(str(e))

dbutils.fs.mkdirs(landing_path)
dbutils.fs.mkdirs(canonical_root)

In [0]:
DEMO_DATA = [
    ("demo", "demo", "manual_demo", "train", "Stocks rallied after the company reported stronger earnings than expected", "positive"),
    ("demo", "demo", "manual_demo", "train", "The bank warned about a possible slowdown in credit demand", "negative"),
    ("demo", "demo", "manual_demo", "validation", "The company maintained its annual forecast during the investor call", "neutral"),
    ("demo", "demo", "manual_demo", "test", "Shares fell after the firm missed revenue expectations", "negative"),
    ("demo", "demo", "manual_demo", "test", "Analysts upgraded the stock because margins improved", "positive"),
]


def safe_ls(path):
    try:
        return dbutils.fs.ls(path)
    except Exception:
        return []


def list_files_recursive(path: str):
    out = []
    for item in safe_ls(path):
        if item.isDir():
            out.extend(list_files_recursive(item.path))
        else:
            out.append(item)
    return out

canonical_files = [f for f in list_files_recursive(canonical_root) if f.path.lower().endswith(".parquet")]

if source_mode == "canonical_parquet" and create_demo_if_empty and len(canonical_files) == 0:
    demo_df = spark.createDataFrame(
        DEMO_DATA,
        "dataset_id string, dataset_label string, source_platform string, split string, text string, label_normalized string",
    ).withColumn("download_batch_id", F.lit("demo")) \
     .withColumn("source_record_hash", F.sha2(F.concat_ws("||", "dataset_label", "split", "text", "label_normalized"), 256))
    demo_path = f"{canonical_root}/demo_financial_sentiment"
    demo_df.write.mode("overwrite").format("parquet").save(demo_path)
    canonical_files = [f for f in list_files_recursive(canonical_root) if f.path.lower().endswith(".parquet")]
    print(f"No había Parquet canónico. Se creó demo en {demo_path}")

In [0]:
def pick_col(df, candidates, default_value=None):
    """
    Retorna la primera columna existente dentro de candidates.
    Si no existe ninguna, retorna un literal con default_value.
    """
    for c in candidates:
        if c in df.columns:
            return F.col(c).cast("string")
    return F.lit(default_value).cast("string")


def build_source_record_hash(*cols):
    """
    Crea una llave técnica estable para controlar idempotencia en Bronze.
    No depende de que la fuente traiga source_record_hash.
    """
    return F.sha2(
        F.concat_ws(
            "||",
            *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in cols]
        ),
        256
    )


if source_mode == "canonical_parquet":
    if len(canonical_files) == 0:
        raise RuntimeError(
            f"No hay Parquet canónico en {canonical_root}. "
            "Ejecuta primero 01_download_source_parquets o habilita create_demo_if_empty=true."
        )

    raw_df = (
        spark.read
        .option("recursiveFileLookup", "true")
        .parquet(canonical_root)
        .withColumn("source_path", F.col("_metadata.file_path"))
    )

    staged_df = (
        raw_df
        .select(
            pick_col(raw_df, ["source_record_hash"], None).alias("source_record_hash_in"),
            pick_col(raw_df, ["text", "sentence", "content", "news", "headline", "Text"]).alias("raw_text"),
            pick_col(raw_df, ["label_normalized", "label", "sentiment", "target", "Label"]).alias("raw_label"),
            pick_col(raw_df, ["split", "source_split", "Split"], None).alias("source_split"),
            pick_col(raw_df, ["dataset_label", "source_dataset", "dataset", "source"], "canonical_parquet").alias("source_dataset"),
            pick_col(raw_df, ["dataset_id", "source_dataset_id"], None).alias("source_dataset_id"),
            pick_col(raw_df, ["source_platform"], "unknown").alias("source_platform"),
            pick_col(raw_df, ["download_batch_id", "source_download_batch_id"], None).alias("source_download_batch_id"),
            F.col("source_path"),
        )
        .withColumn("raw_text", F.trim(F.col("raw_text").cast("string")))
        .withColumn("raw_label", F.trim(F.lower(F.col("raw_label").cast("string"))))
        .withColumn("source_split", F.coalesce(F.col("source_split"), F.lit("unknown")).cast("string"))
        .withColumn("source_dataset", F.coalesce(F.col("source_dataset"), F.lit("canonical_parquet")).cast("string"))
        .withColumn("source_dataset_id", F.coalesce(F.col("source_dataset_id"), F.col("source_dataset")).cast("string"))
        .withColumn("source_platform", F.coalesce(F.col("source_platform"), F.lit("unknown")).cast("string"))
        .withColumn(
            "source_record_hash",
            F.coalesce(
                F.col("source_record_hash_in"),
                build_source_record_hash("raw_text", "raw_label", "source_platform", "source_dataset_id", "source_dataset", "source_split")
            )
        )
        .drop("source_record_hash_in")
    )

else:
    source_files = [f for f in list_files_recursive(landing_path) if f.path.lower().endswith((".csv", ".txt"))]
    if len(source_files) == 0:
        raise RuntimeError(f"No hay archivos CSV/TXT en {landing_path} y source_mode={source_mode}.")

    raw_df = (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .csv([f.path for f in source_files])
        .withColumn("source_path", F.col("_metadata.file_path"))
    )

    staged_df = (
        raw_df
        .select(
            pick_col(raw_df, ["text", "sentence", "content", "news", "headline", "Text"]).alias("raw_text"),
            pick_col(raw_df, ["label", "label_normalized", "sentiment", "target", "Label"]).alias("raw_label"),
            pick_col(raw_df, ["split", "source_split", "Split"], None).alias("source_split"),
            pick_col(raw_df, ["source_dataset", "dataset", "source"], "landing_file").alias("source_dataset"),
            pick_col(raw_df, ["dataset_id", "source_dataset_id"], None).alias("source_dataset_id"),
            F.lit("manual_landing").cast("string").alias("source_platform"),
            F.lit(None).cast("string").alias("source_download_batch_id"),
            F.col("source_path"),
        )
        .withColumn("raw_text", F.trim(F.col("raw_text").cast("string")))
        .withColumn("raw_label", F.trim(F.lower(F.col("raw_label").cast("string"))))
        .withColumn("source_split", F.coalesce(F.col("source_split"), F.lit("unknown")).cast("string"))
        .withColumn("source_dataset", F.coalesce(F.col("source_dataset"), F.lit("landing_file")).cast("string"))
        .withColumn("source_dataset_id", F.coalesce(F.col("source_dataset_id"), F.col("source_dataset")).cast("string"))
        .withColumn(
            "source_record_hash",
            build_source_record_hash("raw_text", "raw_label", "source_platform", "source_dataset_id", "source_dataset", "source_split")
        )
    )

print("Columnas preparadas para Bronze:")
print(staged_df.columns)


In [0]:
# Construcción final de Bronze.
# La columna source_record_hash se calcula aquí para que Bronze sea idempotente
# aunque las fuentes Hugging Face/Kaggle/manuales no la traigan.

required_staged_cols = [
    "source_record_hash", "raw_text", "raw_label", "source_split",
    "source_dataset", "source_dataset_id", "source_platform",
    "source_download_batch_id", "source_path"
]

missing_cols = [c for c in required_staged_cols if c not in staged_df.columns]
if missing_cols:
    raise RuntimeError(f"Faltan columnas requeridas para Bronze: {missing_cols}. Columnas actuales: {staged_df.columns}")

bronze_df = (
    staged_df
    .filter(F.col("raw_text").isNotNull())
    .filter(F.length(F.trim(F.col("raw_text"))) > 0)
    .filter(F.col("raw_label").isNotNull())
    .filter(F.col("source_record_hash").isNotNull())
    .withColumn("bronze_record_id", F.expr("uuid()"))
    .withColumn("ingestion_batch_id", F.lit(batch_id))
    .withColumn("ingestion_ts", F.current_timestamp())
    .select(
        "bronze_record_id",
        "source_record_hash",
        "raw_text",
        "raw_label",
        "source_split",
        "source_dataset",
        "source_dataset_id",
        "source_platform",
        "source_download_batch_id",
        "source_path",
        "ingestion_batch_id",
        "ingestion_ts"
    )
    .dropDuplicates(["source_record_hash"])
)

# Si ya existe una tabla Bronze creada con una versión anterior y no tiene source_record_hash,
# se elimina para evitar el error UNRESOLVED_COLUMN durante el left_anti join.
if spark.catalog.tableExists(bronze_table):
    existing_columns = spark.table(bronze_table).columns
    if "source_record_hash" not in existing_columns:
        print(f"La tabla {bronze_table} existe pero no tiene source_record_hash. Se recreará.")
        spark.sql(f"DROP TABLE IF EXISTS {bronze_table}")

# Carga idempotente: solo inserta hashes que no existan.
if spark.catalog.tableExists(bronze_table):
    existing_hashes = spark.table(bronze_table).select("source_record_hash").distinct()
    bronze_new_df = (
        bronze_df.alias("new")
        .join(existing_hashes.alias("old"), on="source_record_hash", how="left_anti")
    )
else:
    bronze_new_df = bronze_df

rows_written = bronze_new_df.count()

if rows_written > 0:
    (
        bronze_new_df
        .write
        .mode("append")
        .format("delta")
        .option("mergeSchema", "true")
        .saveAsTable(bronze_table)
    )

    source_paths_df = bronze_new_df.groupBy("source_path", "source_dataset").agg(F.count("*").alias("rows_loaded"))
    audit_rows = []
    for row in source_paths_df.collect():
        audit_rows.append((row.source_path, row.source_dataset, None, None, batch_id, int(row.rows_loaded), datetime.utcnow()))

    audit_schema = (
        "source_path string, source_name string, file_size long, modification_time timestamp, "
        "ingestion_batch_id string, rows_loaded long, ingestion_ts timestamp"
    )

    if audit_rows:
        spark.createDataFrame(audit_rows, audit_schema).write.mode("append").saveAsTable(audit_table)

    message = f"Se cargaron {rows_written} registros nuevos en Bronze desde fuentes canónicas. Modo={source_mode}."
else:
    message = f"No hay registros nuevos para cargar en Bronze. Ingesta idempotente. Modo={source_mode}."

status = "SUCCESS"

print(message)
print(f"Tabla Bronze: {bronze_table}")


In [0]:
spark.createDataFrame([
    (batch_id, "02_bronze_ingestion", status, message, int(rows_written), datetime.utcnow())
], "run_id string, step string, status string, message string, rows_written long, event_ts timestamp").write.mode("append").saveAsTable(run_log_table)

try:
    dbutils.jobs.taskValues.set(key="bronze_batch_id", value=batch_id)
    dbutils.jobs.taskValues.set(key="bronze_rows", value=str(rows_written))
except Exception:
    pass

print(message)
display(bronze_new_df.limit(20))